# Personality Type Classifier - Pre-Deployment Testing

This notebook validates the saved models and preprocessing artifacts before deployment. We load a sample from the test dataset, apply the same preprocessing pipeline used during training, and verify that predictions are correct and consistent.

**Models:** gradient_boosting.pkl, svm.pkl  
**Artifacts:** scaler.pkl, label_encoder.pkl, num_imputer.pkl, cat_imputer.pkl  
**Author:** Duwarahavidyan J  
**Date:** 2026-05-07

In [8]:
# Importing Libraries

import pandas as pd
import numpy as np
import joblib 

In [6]:
# Load artifacts
le          = joblib.load('../artifacts/label_encoder.pkl')
scaler      = joblib.load('../artifacts/scaler.pkl')
num_imputer = joblib.load('../artifacts/num_imputer.pkl')
cat_imputer = joblib.load('../artifacts/cat_imputer.pkl')

# Load models
gb_model  = joblib.load('../models/gradient_boosting.pkl')
svm_model = joblib.load('../models/svm.pkl')


In [10]:
# Manual sample input
sample_input = {
    'Time_spent_Alone'         : 7,    
    'Stage_fear'               : 'Yes', 
    'Social_event_attendance'  : 2,    
    'Going_outside'            : 2,    
    'Drained_after_socializing': 'Yes', 
    'Friends_circle_size'      : 3,    
    'Post_frequency'           : 1     
}

sample_df = pd.DataFrame([sample_input])
print("Raw user input:")
sample_df

Raw user input:


,Time_spent_Alone,Stage_fear,Social_event_attendance,Going_outside,Drained_after_socializing,Friends_circle_size,Post_frequency
0,7,Yes,2,2,Yes,3,1


In [12]:
# Apply same preprocessing pipeline as training

#Encode Yes/No
sample_df['Stage_fear']                = sample_df['Stage_fear'].map({'No': 0, 'Yes': 1})
sample_df['Drained_after_socializing'] = sample_df['Drained_after_socializing'].map({'No': 0, 'Yes': 1})

# Impute
cat_cols = ['Stage_fear', 'Drained_after_socializing']
num_cols = [col for col in sample_df.columns if col not in cat_cols]

sample_df[num_cols] = num_imputer.transform(sample_df[num_cols])
sample_df[cat_cols] = cat_imputer.transform(sample_df[cat_cols])

# Scale
sample_scaled = pd.DataFrame(scaler.transform(sample_df), columns=sample_df.columns)


sample_scaled

,Time_spent_Alone,Stage_fear,Social_event_attendance,Going_outside,Drained_after_socializing,Friends_circle_size,Post_frequency
0,0.837553,-0.867787,-0.778776,-0.556885,-0.866906,-0.853511,-0.973091


In [13]:
# Predict with both models
gb_pred  = gb_model.predict(sample_scaled)[0]
svm_pred = svm_model.predict(sample_scaled)[0]

gb_proba  = gb_model.predict_proba(sample_scaled)[0].max()
svm_proba = svm_model.predict_proba(sample_scaled)[0].max()

print(f"\nGradient Boosting : {le.inverse_transform([gb_pred])[0]} ({gb_proba:.2%} confidence)")
print(f"SVM               : {le.inverse_transform([svm_pred])[0]} ({svm_proba:.2%} confidence)")


Gradient Boosting : Introvert (97.01% confidence)
SVM               : Introvert (64.62% confidence)


#### Another Example

In [15]:
# Manual sample input: extrovert profile
sample_input = {
    'Time_spent_Alone'         : 1,     # low alone time
    'Stage_fear'               : 'No',  # no stage fear
    'Social_event_attendance'  : 8,     # high social attendance
    'Going_outside'            : 9,     # goes outside frequently
    'Drained_after_socializing': 'No',  # not drained after socializing
    'Friends_circle_size'      : 12,    # large friend circle
    'Post_frequency'           : 8      # posts frequently
}


#  Create DataFrame
sample_df = pd.DataFrame([sample_input])

# Encode Yes/No
sample_df['Stage_fear']                = sample_df['Stage_fear'].map({'No': 0, 'Yes': 1})
sample_df['Drained_after_socializing'] = sample_df['Drained_after_socializing'].map({'No': 0, 'Yes': 1})

#  Impute
cat_cols = ['Stage_fear', 'Drained_after_socializing']
num_cols = [col for col in sample_df.columns if col not in cat_cols]
sample_df[num_cols] = num_imputer.transform(sample_df[num_cols])
sample_df[cat_cols] = cat_imputer.transform(sample_df[cat_cols])

#  Scale
sample_scaled = pd.DataFrame(scaler.transform(sample_df), columns=sample_df.columns)

#  Predict
gb_pred   = gb_model.predict(sample_scaled)[0]
svm_pred  = svm_model.predict(sample_scaled)[0]
gb_proba  = gb_model.predict_proba(sample_scaled)[0].max()
svm_proba = svm_model.predict_proba(sample_scaled)[0].max()

print(f"  Gradient Boosting : {le.inverse_transform([gb_pred])[0]:<12} ({gb_proba:.2%} confidence)")
print(f"  SVM               : {le.inverse_transform([svm_pred])[0]:<12} ({svm_proba:.2%} confidence)")

  Gradient Boosting : Extrovert    (65.90% confidence)
  SVM               : Extrovert    (93.08% confidence)
